In [108]:
import pandas as pd
import numpy as np
from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache
import os

cache_dir = "./AllenSDK_cache"
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

manifest_path = os.path.join(cache_dir, 'manifest.json')

cache = EcephysProjectCache.from_warehouse(manifest=manifest_path)
sessions = cache.get_session_table()
#for id in session.index:
    


Int64Index([715093703, 719161530, 721123822, 732592105, 737581020, 739448407,
            742951821, 743475441, 744228101, 746083955, 750332458, 750749662,
            751348571, 754312389, 754829445, 755434585, 756029989, 757216464,
            757970808, 758798717, 759883607, 760345702, 760693773, 761418226,
            762120172, 762602078, 763673393, 766640955, 767871931, 768515987,
            771160300, 771990200, 773418906, 774875821, 778240327, 778998620,
            779839471, 781842082, 786091066, 787025148, 789848216, 791319847,
            793224716, 794812542, 797828357, 798911424, 799864342, 816200189,
            819186360, 819701982, 821695405, 829720705, 831882777, 835479236,
            839068429, 839557629, 840012044, 847657808],
           dtype='int64', name='id')

In [12]:
# Load the session data using the specified session_id
session_id = 719161530
output_file = "processed_data_drifting_gratings2.csv"
session = cache.get_session_data(session_id)

# Print available stimulus types to verify loading
stim_types = session.stimulus_presentations['stimulus_name'].unique()
print("Available stimulus types in this session:")
print(stim_types)

Downloading:   0%|          | 0.00/3.07G [00:00<?, ?B/s]

/Users/serbanbantas/anaconda3/envs/envallen/lib/python3.11/site-packages/hdmf/spec/namespace.py:535: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/serbanbantas/anaconda3/envs/envallen/lib/python3.11/site-packages/hdmf/spec/namespace.py:535: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.7.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."


Available stimulus types in this session:
['spontaneous' 'gabors' 'invalid_presentation' 'flashes'
 'drifting_gratings' 'natural_movie_three' 'natural_movie_one'
 'static_gratings' 'natural_scenes']


In [16]:
# Filter stimulus presentations to get only the "drifting_gratings" block
stim_table = session.stimulus_presentations
dg_presentations = stim_table[stim_table['stimulus_name'] == 'drifting_gratings']

# Reset index to simplify repeat counting
dg_presentations = dg_presentations.reset_index(drop=True)

# Drifting gratings are defined by multiple parameters: orientation, temporal frequency, and spatial frequency.
# Create a composite condition id using these three parameters.
dg_presentations['condition_id'] = list(zip(
    dg_presentations['orientation'],
    dg_presentations['temporal_frequency'],
    dg_presentations['spatial_frequency']
))

# Create a mapping for condition order (optional, for metadata)
unique_conditions = dg_presentations['condition_id'].unique()
condition_order = {cond: idx for idx, cond in enumerate(unique_conditions)}
dg_presentations['condition_order'] = dg_presentations['condition_id'].map(condition_order)

# Compute repeat number by grouping by the composite key
dg_presentations['repeat_number'] = dg_presentations.groupby(
    ['orientation', 'temporal_frequency', 'spatial_frequency']
).cumcount()

# Display the beginning and end of the DataFrame for verification
dg_presentations.head()
dg_presentations.tail()

,stimulus_block,start_time,stop_time,x_position,stimulus_name,spatial_frequency,orientation,y_position,frame,color,temporal_frequency,phase,contrast,size,duration,stimulus_condition_id,condition_id,condition_order,repeat_number
623,7.0,5383.300753,5385.302413,null,drifting_gratings,0.04,45.0,null,null,null,15.0,"[42423.86666667, 42423.86666667]",0.8,"[250.0, 250.0]",2.00166,267.0,"(45.0, 15.0, 0.04)",21,14
624,7.0,5386.303267,5388.304927,null,drifting_gratings,0.04,0.0,null,null,null,1.0,"[42423.86666667, 42423.86666667]",0.8,"[250.0, 250.0]",2.00166,262.0,"(0.0, 1.0, 0.04)",16,13
625,7.0,5389.305763,5391.307423,null,drifting_gratings,0.04,180.0,null,null,null,2.0,"[42423.86666667, 42423.86666667]",0.8,"[250.0, 250.0]",2.00166,256.0,"(180.0, 2.0, 0.04)",10,14
626,7.0,5392.308287,5394.309947,null,drifting_gratings,0.04,315.0,null,null,null,8.0,"[42423.86666667, 42423.86666667]",0.8,"[250.0, 250.0]",2.00166,260.0,"(315.0, 8.0, 0.04)",14,14
627,7.0,5395.310763,5397.312443,null,drifting_gratings,0.04,90.0,null,null,null,8.0,"[42423.86666667, 42423.86666667]",0.8,"[250.0, 250.0]",2.00168,273.0,"(90.0, 8.0, 0.04)",27,14


In [19]:
print(session.units.columns)

Index(['waveform_PT_ratio', 'waveform_amplitude', 'amplitude_cutoff',
       'cluster_id', 'cumulative_drift', 'd_prime', 'firing_rate',
       'isi_violations', 'isolation_distance', 'L_ratio', 'local_index',
       'max_drift', 'nn_hit_rate', 'nn_miss_rate', 'peak_channel_id',
       'presence_ratio', 'waveform_recovery_slope',
       'waveform_repolarization_slope', 'silhouette_score', 'snr',
       'waveform_spread', 'waveform_velocity_above', 'waveform_velocity_below',
       'waveform_duration', 'filtering', 'probe_channel_number',
       'probe_horizontal_position', 'probe_id', 'probe_vertical_position',
       'structure_acronym', 'ecephys_structure_id',
       'ecephys_structure_acronym', 'anterior_posterior_ccf_coordinate',
       'dorsal_ventral_ccf_coordinate', 'left_right_ccf_coordinate',
       'probe_description', 'location', 'probe_sampling_rate',
       'probe_lfp_sampling_rate', 'probe_has_lfp_data'],
      dtype='object')


In [39]:
# Debug: Print available columns and a few example keys from spike_times
print("Units columns:", session.units.columns)
print("First few keys in session.spike_times:", list(session.spike_times.keys())[:5])
if len(session.spike_times) > 0:
    example_key = list(session.spike_times.keys())[0]
    print("Type of a spike_times key:", type(example_key))

# Here, 'isi_violations' less than 0.02 is used as a quality metric.
visp_units = session.units[(session.units['structure_acronym'] == 'VISp') &
                           (session.units['isi_violations'] < 0.02)]

print(f"Number of VISp units selected: {len(visp_units)}")

Units columns: Index(['waveform_PT_ratio', 'waveform_amplitude', 'amplitude_cutoff',
       'cluster_id', 'cumulative_drift', 'd_prime', 'firing_rate',
       'isi_violations', 'isolation_distance', 'L_ratio', 'local_index',
       'max_drift', 'nn_hit_rate', 'nn_miss_rate', 'peak_channel_id',
       'presence_ratio', 'waveform_recovery_slope',
       'waveform_repolarization_slope', 'silhouette_score', 'snr',
       'waveform_spread', 'waveform_velocity_above', 'waveform_velocity_below',
       'waveform_duration', 'filtering', 'probe_channel_number',
       'probe_horizontal_position', 'probe_id', 'probe_vertical_position',
       'structure_acronym', 'ecephys_structure_id',
       'ecephys_structure_acronym', 'anterior_posterior_ccf_coordinate',
       'dorsal_ventral_ccf_coordinate', 'left_right_ccf_coordinate',
       'probe_description', 'location', 'probe_sampling_rate',
       'probe_lfp_sampling_rate', 'probe_has_lfp_data'],
      dtype='object')
First few keys in session.spik

In [68]:
# Define the spike counting window (from stimulus onset to 500 ms later) (doesn't help in the end)
window_start = 0.0  # relative to stimulus onset
window_end = 0.5    # count spikes until 500 ms post onset

results = []

for _, unit in visp_units.iterrows():
    unit_id = int(unit.name)
    spike_times = session.spike_times[unit_id]
    
    for _, stim in dg_presentations.iterrows():
        if pd.isnull(stim['orientation']):
            continue
            
        stim_start = stim['start_time']
        spikes_in_window = np.count_nonzero(
            (spike_times >= stim_start + window_start) &
            (spike_times < stim_start + window_end)
        )
        results.append({
            'unit_id': unit_id,
            'orientation': stim['orientation'],
            'temporal_frequency': stim['temporal_frequency'],
            'spatial_frequency': stim['spatial_frequency'],
            'repeat_number': stim['repeat_number'],
            'stimulus_start_time': stim_start,
            'spike_count': spikes_in_window,
            'session_id': session_id
        })

print("Spike counting complete!")

Spike counting complete!


In [60]:
print("Drifting Gratings Presentations:", dg_presentations.shape)
print("VISp Units:", visp_units.shape)
print(session.stimulus_presentations['stimulus_name'].unique())


Drifting Gratings Presentations: (628, 19)
VISp Units: (21, 40)
['spontaneous' 'gabors' 'invalid_presentation' 'flashes'
 'drifting_gratings' 'natural_movie_three' 'natural_movie_one'
 'static_gratings' 'natural_scenes']


In [72]:
df_results = pd.DataFrame(results)
df_results.to_csv(output_file, index=False)
print(f"Data saved to {output_file}")


Data saved to processed_data_drifting_gratings2.csv
